# Widget Communication Patterns

This notebook explores different patterns for inter-widget communication, especially for static HTML exports.

In [1]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path().absolute().parent))

from widgets.counter_widget import CounterWidget
from widgets.linked_counter import LinkedCounterWidget
import ipywidgets as widgets
from IPython.display import display, HTML

## Pattern 1: Direct Model Linking

Widgets can directly access each other's Backbone models through a global registry:

In [2]:
# Create source widget
source = CounterWidget(
    value=10,
    label="Source Widget",
    widget_id="source_1"
)

# Create linked widget that mirrors the source
mirror = LinkedCounterWidget(
    value=0,
    label="Mirror Widget",
    widget_id="mirror_1",
    link_to="source_1",
    link_mode="mirror"
)

display(HTML("<h3>Direct Model Linking</h3>"))
display(HTML("<p>Change the source widget and watch the mirror update automatically:</p>"))
display(widgets.HBox([source, mirror]))

## Pattern 2: Event-Based Communication

Widgets can communicate through custom events dispatched on a global event bus:

In [3]:
display(HTML("""
<h3>Event-Based Communication</h3>
<p>These widgets communicate through custom events. Open the browser console to see event logs.</p>
<script>
// Listen to global widget events
if (window.__widgetEvents) {
    window.__widgetEvents.addEventListener('counter-changed', (e) => {
        console.log('Widget event:', e.detail);
    });
}
</script>
"""))

# Create multiple widgets that emit events
event_source = CounterWidget(
    value=0,
    label="Event Emitter",
    widget_id="event_source"
)

event_listener = LinkedCounterWidget(
    value=0,
    label="Event Listener",
    widget_id="event_listener",
    link_to="event_source",
    link_mode="mirror"
)

display(widgets.HBox([event_source, event_listener]))

## Pattern 3: Computed Values

Widgets can compute values based on multiple other widgets:

In [4]:
# Create input widgets
input1 = CounterWidget(
    value=5,
    label="Input 1",
    widget_id="input_1"
)

input2 = CounterWidget(
    value=3,
    label="Input 2",
    widget_id="input_2"
)

# Create computed widgets
sum_widget = LinkedCounterWidget(
    value=0,
    label="Sum (Input 1 + Input 2)",
    widget_id="sum_widget",
    link_to="input_1",
    link_mode="sum"
)

diff_widget = LinkedCounterWidget(
    value=10,
    label="Difference (10 - Input 2)",
    widget_id="diff_widget",
    link_to="input_2",
    link_mode="diff"
)

display(HTML("<h3>Computed Values</h3>"))
display(HTML("<p>The bottom widgets compute values based on the top widgets:</p>"))
display(widgets.VBox([
    widgets.HBox([input1, input2]),
    widgets.HBox([sum_widget, diff_widget])
]))

## Pattern 4: Broadcast Communication

One widget can broadcast to multiple listeners:

In [5]:
# Create broadcaster
broadcaster = CounterWidget(
    value=100,
    label="Broadcaster",
    widget_id="broadcaster"
)

# Create multiple listeners
listeners = []
for i in range(3):
    listener = LinkedCounterWidget(
        value=0,
        label=f"Listener {i+1}",
        widget_id=f"listener_{i+1}",
        link_to="broadcaster",
        link_mode="mirror" if i == 0 else "sum" if i == 1 else "diff"
    )
    listeners.append(listener)

display(HTML("<h3>Broadcast Pattern</h3>"))
display(HTML("<p>One broadcaster, multiple listeners with different modes:</p>"))
display(widgets.VBox([
    broadcaster,
    widgets.HBox(listeners)
]))

## Pattern 5: Chain Communication

Widgets can form a chain where each depends on the previous:

In [6]:
# Create a chain of widgets
chain_widgets = []

# First widget in chain
chain_widgets.append(CounterWidget(
    value=1,
    label="Chain Start",
    widget_id="chain_0"
))

# Create chain links
for i in range(3):
    widget = LinkedCounterWidget(
        value=i * 10,
        label=f"Chain Link {i+1}",
        widget_id=f"chain_{i+1}",
        link_to=f"chain_{i}",
        link_mode="sum"
    )
    chain_widgets.append(widget)

display(HTML("<h3>Chain Communication</h3>"))
display(HTML("<p>Each widget depends on the previous one (sum mode):</p>"))
for w in chain_widgets:
    display(w)

## Testing Static Export Compatibility

These patterns are designed to work in static HTML exports. Let's create a test case:

In [7]:
display(HTML("""
<h3>Static Export Test</h3>
<p>This section tests widget functionality for static exports:</p>
<ul>
    <li>✅ Widgets should maintain their state</li>
    <li>✅ Inter-widget communication should work</li>
    <li>✅ No Python kernel required</li>
</ul>
"""))

# Create test widgets with specific initial states
static_source = CounterWidget(
    value=42,
    label="Static Source",
    widget_id="static_source"
)

static_mirror = LinkedCounterWidget(
    value=0,
    label="Static Mirror",
    widget_id="static_mirror",
    link_to="static_source",
    link_mode="mirror"
)

display(widgets.HBox([static_source, static_mirror]))

display(HTML("""
<div style="margin-top: 20px; padding: 10px; background: #f0f0f0; border-radius: 5px;">
    <b>Instructions for testing static export:</b>
    <ol>
        <li>Build the book: <code>myst build --html</code></li>
        <li>Open the static HTML file</li>
        <li>Click the buttons on the Static Source widget</li>
        <li>Verify that the Static Mirror widget updates</li>
    </ol>
</div>
"""))

## Summary

We've demonstrated several communication patterns:

1. **Direct Model Linking** - Widgets access each other's Backbone models
2. **Event-Based** - Custom events on a global event bus
3. **Computed Values** - Widgets compute values from others
4. **Broadcast** - One-to-many communication
5. **Chain** - Sequential dependencies

### Key Insights

- All patterns work without Python kernel in static exports
- Global registries enable widget discovery
- Event systems provide loose coupling
- Backbone models handle state synchronization

These patterns form the foundation for building complex, interactive geospatial applications.